# Step 1 — Wikidata Enrichment

Queries Wikidata SPARQL for biographical matches on each person in the database, filtering by name, occupation (journalist, editor, publisher), and approximate active date range.

**Expected hit rate:** Low. Most county-level editors and small-town newspaper owners from the 1870s–1880s are not represented in Wikidata. The value of this step is in the hits that do occur: for matched individuals, we harvest birth/death dates and places, occupations, and external identifier links — VIAF, LCCN, Find a Grave, and GND. These identifiers are valuable pointers for subsequent search steps (HathiTrust, Internet Archive, etc.).

**FamilySearch:** Integration with FamilySearch is noted as a future extension. It requires OAuth2 authentication and per-user developer credentials. A commented stub is provided at the end of this notebook.

In [2]:
import sys, json, time
from pathlib import Path
import requests
from tqdm import tqdm
sys.path.insert(0, str(Path('.')))
from db import get_connection, set_progress, get_progress, pending_persons, store_enrichment, get_all_persons

conn = get_connection()
STEP = '1_wikidata'
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
HEADERS = {"User-Agent": "RailroadTiesPipeline/1.0 (historical research; contact via GitHub)"}

## Wikidata query function

In [3]:
def sparql_query(query, retries=3):
    """Run a SPARQL query against Wikidata. Returns list of result bindings."""
    for attempt in range(retries):
        try:
            resp = requests.get(
                WIKIDATA_ENDPOINT,
                params={"query": query, "format": "json"},
                headers=HEADERS,
                timeout=30
            )
            resp.raise_for_status()
            return resp.json().get("results", {}).get("bindings", [])
        except requests.exceptions.Timeout:
            print(f"  Timeout (attempt {attempt+1})")
        except requests.exceptions.HTTPError as e:
            if resp.status_code == 429:
                wait = 60 * (attempt + 1)
                print(f"  Rate limited, waiting {wait}s")
                time.sleep(wait)
            else:
                print(f"  HTTP error: {e}")
                break
        except Exception as e:
            print(f"  Error: {e}")
        time.sleep(5 * (attempt + 1))
    return []


def search_person_wikidata(canonical_name, first_year=None, last_year=None):
    """
    Search Wikidata for a person by name. Returns list of candidate dicts.
    Uses a broad label search, then filters by occupation if needed.
    """
    # Escape quotes in name
    safe_name = canonical_name.replace('"', '\\"')
    
    query = f"""
    SELECT DISTINCT ?item ?itemLabel ?birthDate ?deathDate ?birthPlaceLabel ?deathPlaceLabel
                    ?occupationsLabel ?viaf ?lccn ?findagrave ?gndId
    WHERE {{
      ?item wikibase:sitelinks ?sitelinks .
      ?item rdfs:label "{safe_name}"@en .
      ?item wdt:P31 wd:Q5 .  # instance of: human
      OPTIONAL {{ ?item wdt:P569 ?birthDate. }}
      OPTIONAL {{ ?item wdt:P570 ?deathDate. }}
      OPTIONAL {{ ?item wdt:P19 ?birthPlace. }}
      OPTIONAL {{ ?item wdt:P20 ?deathPlace. }}
      OPTIONAL {{ ?item wdt:P106 ?occupations. }}
      OPTIONAL {{ ?item wdt:P214 ?viaf. }}
      OPTIONAL {{ ?item wdt:P244 ?lccn. }}
      OPTIONAL {{ ?item wdt:P535 ?findagrave. }}
      OPTIONAL {{ ?item wdt:P227 ?gndId. }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 10
    """
    results = sparql_query(query)
    
    candidates = []
    seen_qids = set()
    for r in results:
        qid_url = r.get("item", {}).get("value", "")
        qid = qid_url.rsplit("/", 1)[-1] if qid_url else None
        if qid in seen_qids:
            continue
        seen_qids.add(qid)
        
        def val(key):
            return r.get(key, {}).get("value", None)
        
        # Parse birth/death years from ISO dates
        def parse_year(date_str):
            if not date_str:
                return None
            try:
                return int(date_str[:4])
            except (ValueError, TypeError):
                return None
        
        birth_yr = parse_year(val("birthDate"))
        death_yr = parse_year(val("deathDate"))
        
        # Filter by approximate date range if known
        if first_year and birth_yr:
            # Person should have been born before their first active year
            if birth_yr > first_year:
                continue
        if last_year and death_yr:
            # Person should have died after or during their active period
            if death_yr < (first_year or 1860):
                continue
        
        external_ids = {}
        for id_key in ("viaf", "lccn", "findagrave", "gndId"):
            v = val(id_key)
            if v:
                external_ids[id_key] = v
        
        candidates.append({
            "qid": qid,
            "label": val("itemLabel"),
            "birth_year": birth_yr,
            "death_year": death_yr,
            "birth_place": val("birthPlaceLabel"),
            "death_place": val("deathPlaceLabel"),
            "occupations": val("occupationsLabel"),
            "external_ids": external_ids,
            "raw": r,
        })
    
    return candidates

## Run enrichment loop

In [4]:
DELAY_BETWEEN = 1.2  # seconds between queries (Wikidata asks for >=1s)
DRY_RUN = False       # set True to test without writing to DB

todo = pending_persons(conn, STEP)
print(f"{len(todo)} persons pending for step '{STEP}'")
print("(Set DRY_RUN=True in the cell above to preview without writing)\n")

hit_count = 0
for research_id in tqdm(todo):
    person = conn.execute(
        "SELECT canonical_name, first_active_year, last_active_year FROM persons WHERE research_id=?",
        (research_id,)
    ).fetchone()
    
    name       = person["canonical_name"]
    first_yr   = person["first_active_year"]
    last_yr    = person["last_active_year"]
    
    try:
        candidates = search_person_wikidata(name, first_yr, last_yr)
        
        if candidates and not DRY_RUN:
            # Take the best candidate (first after date filtering)
            best = candidates[0]
            store_enrichment(
                conn,
                research_id=research_id,
                source='wikidata',
                birth_year=best.get("birth_year"),
                death_year=best.get("death_year"),
                birth_place=best.get("birth_place"),
                death_place=best.get("death_place"),
                occupations=[best["occupations"]] if best.get("occupations") else [],
                external_ids=best.get("external_ids", {}),
                wikidata_qid=best.get("qid"),
                raw_data={"candidates": candidates},
            )
            hit_count += 1
        
        if not DRY_RUN:
            set_progress(conn, research_id, STEP, 'done',
                         result_count=len(candidates))
        else:
            if candidates:
                print(f"  DRY RUN [{research_id}] {name}: {candidates[0]['qid']} ({candidates[0]['birth_year']}\u2013{candidates[0]['death_year']})")
        
    except Exception as e:
        print(f"  Error for {name}: {e}")
        if not DRY_RUN:
            set_progress(conn, research_id, STEP, 'failed', error_msg=str(e))
    
    time.sleep(DELAY_BETWEEN)

print(f"\nDone. Wikidata hits: {hit_count}/{len(todo)}")

570 persons pending for step '1_wikidata'
(Set DRY_RUN=True in the cell above to preview without writing)



100%|██████████| 570/570 [22:46<00:00,  2.40s/it]


Done. Wikidata hits: 122/570


## Results summary

In [5]:
rows = conn.execute(
    """SELECT p.canonical_name, e.wikidata_qid, e.birth_year, e.death_year, e.birth_place, e.external_ids
       FROM enrichment e JOIN persons p ON e.research_id = p.research_id
       WHERE e.source='wikidata'
       ORDER BY p.canonical_name"""
).fetchall()

print(f"Total Wikidata enrichments: {len(rows)}\n")
print(f"{'Name':<35} {'QID':<12} {'Born':<6} {'Died':<6} {'Birth Place'}")
print("-" * 85)
for r in rows[:40]:
    ext = json.loads(r['external_ids'] or '{}')
    ext_str = ', '.join(f"{k}:{v[:10]}" for k, v in list(ext.items())[:2])
    print(f"{r['canonical_name']:<35} {r['wikidata_qid'] or '':<12} {r['birth_year'] or '':<6} {r['death_year'] or '':<6} {(r['birth_place'] or '')[:30]}")

Total Wikidata enrichments: 122

Name                                QID          Born   Died   Birth Place
-------------------------------------------------------------------------------------
Abigail Scott Duniway               Q4667713     1834   1915   Groveland
Alfred Doten                        Q55720580    1829   1903   Plymouth
Bayard Taylor                       Q812105      1825   1878   Kennett Square
C. Wilkinson                        Q76017320                  
Charles A. Storke                   Q5074920     1847   1936   Branchport
Charles Cole                        Q134601810                 
Charles E. Cassell                  Q5077061     1838   1916   Portsmouth
Charles E. Cassell                  Q5077061     1838   1916   Portsmouth
Charles E. Chapin                   Q5076188     1858   1930   
Charles F. Jenkins                  Q5077527                   Towns County
Charles F. Scott                    Q1064330     1864   1944   Athens
Charles H. Davidson    

## Note on FamilySearch

FamilySearch requires OAuth2 authentication and per-user credentials (register at familysearch.org/developers). When credentials are available, add them to `.env` as `FAMILYSEARCH_TOKEN` and uncomment the cell below. The FamilySearch Search API endpoint is `https://api.familysearch.org/platform/tree/search` with parameters `q.givenName`, `q.surname`, `q.birthLikeYear`, `q.residencePlace`.

In [ ]:
# ---------------------------------------------------------------------------
# FamilySearch stub — requires OAuth token in .env as FAMILYSEARCH_TOKEN
# ---------------------------------------------------------------------------
# import os
# from dotenv import load_dotenv
# load_dotenv()
# FS_TOKEN = os.getenv("FAMILYSEARCH_TOKEN")
#
# def search_familysearch(given, surname, birth_year=None, state=None):
#     if not FS_TOKEN:
#         print("No FAMILYSEARCH_TOKEN in .env — skipping")
#         return []
#     params = {
#         "q.givenName": given,
#         "q.surname": surname,
#         "count": 10,
#     }
#     if birth_year:
#         params["q.birthLikeYear"] = str(birth_year)
#         params["q.birthLikeYearRange"] = "5"
#     if state:
#         params["q.residencePlace"] = state
#     headers = {
#         "Authorization": f"Bearer {FS_TOKEN}",
#         "Accept": "application/x-fs-v1+json",
#     }
#     resp = requests.get(
#         "https://api.familysearch.org/platform/tree/search",
#         params=params, headers=headers, timeout=20
#     )
#     resp.raise_for_status()
#     return resp.json().get("entries", [])